In [ ]:
!pip install yfinance requests

In [ ]:
# ======================================================================
# PRISM v2.3 — Fixed Multi-Source Fetch
# ======================================================================
# Cell 1: !pip install yfinance requests
# Cell 2: Paste this entire script
# Cell 3: Download CSVs
# ======================================================================

import pandas as pd
import numpy as np
import yfinance as yf
import requests
import time
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")

# ======================================================================
#  SECTION 1: WATCHLIST + API KEYS
# ======================================================================

TICKERS = [
    "AVGO","MU","TSM", "ASML","000660.KS",
    "APP","RBRK",
    "NFLX","RDDT",
    "UBER","SE",
    "LLY", "TMDX",
    "GOOG", "AMZN","META","NVDA","MSFT",
    "KNSL","CRCL","AXP","DNP.WA", "MCO","BRK.B",
    "FICO", "V", "MA","NU","CVNA","NBIS",
]

FMP_API_KEY = "YOUR_FMP_KEY_HERE"
FINNHUB_API_KEY = "YOUR_FINNHUB_KEY_HERE"

# ======================================================================
#  SECTION 2: CONFIGURATION
# ======================================================================

CONFIG = {
    "weight_growth": 0.35, "weight_value": 0.25,
    "weight_momentum": 0.20, "weight_resilience": 0.20,
    "blend_2026": 0.75, "blend_2027": 0.25, "growth_cap": 2.00,
    "edge_size": 10, "fragile_threshold": 3.0,
    "growth_overrides": {},
    "force_include": [], "force_exclude": [],
    "output_dir": ".",
}

# ======================================================================
#  SECTION 3: MULTI-SOURCE DATA FETCH (v2.3 fixed)
# ======================================================================

def fetch_fmp_estimates(symbol, api_key):
    """FMP stable endpoint for analyst estimates."""
    try:
        url = (f"https://financialmodelingprep.com/stable/analyst-estimates"
               f"?symbol={symbol}&period=annual&page=0&limit=10&apikey={api_key}")
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        if isinstance(data, dict) and ("Error" in str(data) or "message" in data):
            return None
        if not isinstance(data, list) or len(data) == 0:
            return None
        return data
    except:
        return None

def fetch_finnhub_estimates(symbol, api_key):
    """Finnhub EPS estimates (may require paid plan)."""
    try:
        url = f"https://finnhub.io/api/v1/stock/eps-estimate?symbol={symbol}&freq=annual&token={api_key}"
        resp = requests.get(url, timeout=10)
        if resp.status_code != 200:
            return None
        data = resp.json()
        if not data or "data" not in data or len(data.get("data",[])) == 0:
            return None
        return data["data"]
    except:
        return None

def map_fy_to_calendar(date_str):
    try:
        dt = datetime.strptime(date_str[:10], "%Y-%m-%d")
        return dt.year - 1 if dt.month <= 5 else dt.year
    except:
        return None

def calc_growth_from_fmp(fmp_data):
    if not fmp_data:
        return np.nan, np.nan, np.nan, "no_data"
    cal_eps = {}
    for entry in fmp_data:
        cy = map_fy_to_calendar(entry.get("date", ""))
        eps = entry.get("estimatedEpsAvg")
        if cy and eps is not None:
            cal_eps[cy] = eps
    e25, e26, e27 = cal_eps.get(2025), cal_eps.get(2026), cal_eps.get(2027)
    g26 = g27 = np.nan
    if e25 and e26 and e25 > 0: g26 = (e26 / e25) - 1
    elif e25 and e26 and e25 < 0 and e26 > 0: g26 = abs(e26 - e25) / abs(e25)
    if e26 and e27 and e26 > 0: g27 = (e27 / e26) - 1
    elif e26 and e27 and e26 < 0 and e27 > 0: g27 = abs(e27 - e26) / abs(e26)
    return g26, g27, e26, "ok"

def calc_growth_from_finnhub(fh_data):
    if not fh_data:
        return np.nan, np.nan
    cal_eps = {}
    for entry in fh_data:
        cy = map_fy_to_calendar(entry.get("date", ""))
        eps = entry.get("epsAvg")
        if cy and eps is not None:
            cal_eps[cy] = eps
    e25, e26, e27 = cal_eps.get(2025), cal_eps.get(2026), cal_eps.get(2027)
    g26 = (e26 / e25) - 1 if (e25 and e26 and e25 > 0) else np.nan
    g27 = (e27 / e26) - 1 if (e26 and e27 and e26 > 0) else np.nan
    return g26, g27

def validate_growth(fmp_g, fh_g):
    has_fmp, has_fh = pd.notna(fmp_g), pd.notna(fh_g)
    if has_fmp and has_fh:
        if fmp_g == 0 and fh_g == 0: return 0.0, "HIGH"
        avg = (abs(fmp_g) + abs(fh_g)) / 2
        if avg == 0: return fmp_g, "MED"
        diff = abs(fmp_g - fh_g) / max(avg, 0.01)
        if diff <= 0.15: return fmp_g, "HIGH"
        elif diff <= 0.30: return (fmp_g + fh_g) / 2, "MED"
        else: return fmp_g, "LOW"
    elif has_fmp: return fmp_g, "MED"
    elif has_fh: return fh_g, "MED"
    else: return np.nan, "NONE"

def fetch_all_data(tickers, cfg, fmp_key, fh_key):
    print(f"\n{'='*90}")
    print(f"  PRISM v2.3 — Multi-Source Data Fetch")
    print(f"{'='*90}")
    print(f"  Tickers: {len(tickers)} | Sources: FMP (stable) + Finnhub + yfinance\n")

    rows = []
    quality_log = []

    # Source availability — try first 3 tickers before disabling
    fmp_enabled = True
    fh_enabled = True
    fmp_fail_count = 0
    fh_fail_count = 0

    for i, sym in enumerate(tickers):
        # --- yfinance: price, beta, market cap, targets ---
        try:
            tk = yf.Ticker(sym)
            info = tk.info
            price = info.get("currentPrice") or info.get("regularMarketPrice", np.nan)
            mcap = info.get("marketCap", np.nan)
            beta = info.get("beta", np.nan)
            high52 = info.get("fiftyTwoWeekHigh", np.nan)
            low52 = info.get("fiftyTwoWeekLow", np.nan)
            industry = info.get("industry", "Unknown")
            target_mean = info.get("targetMeanPrice", np.nan)
        except:
            tk = None
            price = mcap = beta = high52 = low52 = np.nan
            industry = "Unknown"
            target_mean = np.nan

        avg_outcome = ((target_mean / price) - 1) if (pd.notna(price) and pd.notna(target_mean) and price > 0) else 0.0

        # --- FMP: analyst estimates (primary growth source) ---
        fmp_g26 = fmp_g27 = fmp_fwd_eps = np.nan
        fmp_status = ""
        if fmp_enabled:
            fmp_data = fetch_fmp_estimates(sym, fmp_key)
            if fmp_data is not None:
                fmp_g26, fmp_g27, fmp_fwd_eps, fmp_status = calc_growth_from_fmp(fmp_data)
                if i == 0:
                    print(f"  ✅ FMP connected — got {len(fmp_data)} entries for {sym}")
            else:
                fmp_fail_count += 1
                if i < 3:
                    print(f"  ⚠️  FMP returned nothing for {sym} (fail {fmp_fail_count}/3)")
                if fmp_fail_count >= 3:
                    print(f"  ❌ FMP failed on first 3 tickers — disabling FMP")
                    fmp_enabled = False

        # --- Finnhub: EPS estimates (validation) ---
        fh_g26 = fh_g27 = np.nan
        if fh_enabled:
            fh_data = fetch_finnhub_estimates(sym, fh_key)
            if fh_data is not None:
                fh_g26, fh_g27 = calc_growth_from_finnhub(fh_data)
                if i == 0:
                    print(f"  ✅ Finnhub connected — got {len(fh_data)} entries for {sym}")
            else:
                fh_fail_count += 1
                if i < 3:
                    print(f"  ⚠️  Finnhub returned nothing for {sym} (fail {fh_fail_count}/3)")
                if fh_fail_count >= 3:
                    print(f"  ❌ Finnhub failed on first 3 tickers — disabling (likely premium-only)")
                    fh_enabled = False

        # --- Cross-validate FMP vs Finnhub ---
        g2026, c26 = validate_growth(fmp_g26, fh_g26)
        g2027, c27 = validate_growth(fmp_g27, fh_g27)

        # --- yfinance fallback for growth ---
        if pd.isna(g2026) or pd.isna(g2027):
            try:
                ge = tk.get_growth_estimates() if tk else None
                if ge is not None and not ge.empty:
                    col = ge[sym] if sym in ge.columns else (ge.iloc[:, 0] if len(ge.columns) > 0 else pd.Series(dtype=float))
                    yf_cy = col.get("0y", np.nan)
                    yf_ny = col.get("+1y", np.nan)
                    if pd.notna(yf_cy) and abs(yf_cy) > 5: yf_cy /= 100
                    if pd.notna(yf_ny) and abs(yf_ny) > 5: yf_ny /= 100
                    if pd.isna(g2026) and pd.notna(yf_cy): g2026 = yf_cy; c26 = "YF"
                    if pd.isna(g2027) and pd.notna(yf_ny): g2027 = yf_ny; c27 = "YF"
            except:
                pass

        # --- Manual overrides ---
        if sym in cfg["growth_overrides"]:
            g2026, g2027 = cfg["growth_overrides"][sym]
            c26 = c27 = "OVERRIDE"

        # --- Default missing ---
        if pd.isna(g2026): g2026 = 0.04; c26 = "DEFAULT"
        if pd.isna(g2027): g2027 = 0.04; c27 = "DEFAULT"

        # --- Forward P/E: FMP first, then yfinance ---
        fwd_pe = np.nan
        if pd.notna(fmp_fwd_eps) and fmp_fwd_eps > 0 and pd.notna(price) and price > 0:
            fwd_pe = price / fmp_fwd_eps
        else:
            try:
                ee = tk.get_earnings_estimate() if tk else None
                if ee is not None and not ee.empty and '0y' in ee.index and 'avg' in ee.columns:
                    yf_eps = ee.loc['0y', 'avg']
                    if pd.notna(yf_eps) and yf_eps > 0: fwd_pe = price / yf_eps
            except:
                pass
            if pd.isna(fwd_pe):
                try: fwd_pe = info.get("forwardPE", np.nan)
                except: pass

        quality_log.append({"Stock": sym, "G26_conf": c26, "G27_conf": c27,
            "FMP_g26": fmp_g26, "FH_g26": fh_g26, "Final_g26": g2026,
            "FMP_g27": fmp_g27, "FH_g27": fh_g27, "Final_g27": g2027})
        rows.append({"Stock": sym, "Price": price, "Market Cap": mcap, "Beta": beta,
            "52 Week High": high52, "52 Week Low": low52, "P/E.1": fwd_pe,
            "Industry": industry, "Target Mean": target_mean,
            "Average Outcome": avg_outcome, "2026 Growth Rate": g2026, "2027 Growth Rate": g2027})

        ic = {"HIGH":"✅H","MED":"⚠M","LOW":"🚩L","YF":"📊Y","DEFAULT":"❓D","OVERRIDE":"🔧O","NONE":"❓?"}
        pe_s = f"{fwd_pe:>6.1f}" if pd.notna(fwd_pe) else "   N/A"
        b_s = f"{beta:>5.2f}" if pd.notna(beta) else "  N/A"
        print(f"  [{i+1:2d}/{len(tickers)}] {sym:>6} | G26:{g2026:>+6.0%} [{ic.get(c26,'?'):>3}] | G27:{g2027:>+6.0%} [{ic.get(c27,'?'):>3}] | PE:{pe_s} | B:{b_s}")
        time.sleep(0.3)

    df = pd.DataFrame(rows)
    qdf = pd.DataFrame(quality_log)

    # Data Quality Scorecard
    print(f"\n{'='*90}")
    print(f"  DATA QUALITY SCORECARD")
    print(f"{'='*90}")
    print(f"  Sources active: FMP={'YES' if fmp_enabled else 'NO (disabled)'} | Finnhub={'YES' if fh_enabled else 'NO (disabled)'} | yfinance=YES")
    for label, col in [("2026 Growth", "G26_conf"), ("2027 Growth", "G27_conf")]:
        ct = qdf[col].value_counts(); t = len(qdf)
        h, m, l, y, d, o = ct.get("HIGH",0), ct.get("MED",0), ct.get("LOW",0), ct.get("YF",0), ct.get("DEFAULT",0), ct.get("OVERRIDE",0)
        pct = (h+m+o)/t*100
        print(f"  {label}: ✅HIGH:{h} ⚠MED:{m} 🚩LOW:{l} 📊YF:{y} ❓DEFAULT:{d} 🔧OVR:{o} | {pct:.0f}% reliable")
    bad = qdf[(qdf["G26_conf"].isin(["LOW","DEFAULT","NONE"]))|(qdf["G27_conf"].isin(["LOW","DEFAULT","NONE"]))]
    if len(bad) > 0:
        print(f"\n  ⚠️  Stocks needing overrides: {bad['Stock'].tolist()}")
    return df, qdf

# ======================================================================
#  SECTION 4: SHOCK SCENARIOS
# ======================================================================

SHOCK_SCENARIOS = {
    "Rate Shock":      {"probability":0.10,"market_dd":-0.25,"valuation_matters":True,"mean_reversion":True,"profit_matters":True,"liquidity_matters":False,"sector_multipliers":{"Semiconductor":1.2,"Hardware":1.1,"Software":1.3,"Cybersecurity":1.3,"Cloud Infra":1.4,"Streaming":1.1,"E-commerce":1.2,"Financials":0.8,"Banking":0.9,"Pharma":0.7,"Healthcare":0.7,"Insurance":0.7,"Oil Services":0.8,"Auto":0.9}},
    "COVID Crash":     {"probability":0.05,"market_dd":-0.34,"valuation_matters":False,"mean_reversion":False,"profit_matters":True,"liquidity_matters":True,"sector_multipliers":{"Semiconductor":1.0,"Hardware":1.0,"Software":0.9,"Cybersecurity":0.9,"Cloud Infra":0.9,"Streaming":0.7,"E-commerce":0.8,"Financials":1.1,"Banking":1.1,"Pharma":0.8,"Healthcare":0.8,"Insurance":1.0,"Oil Services":1.5,"Auto":1.2}},
    "AI Bubble Pop":   {"probability":0.15,"market_dd":-0.15,"valuation_matters":True,"mean_reversion":True,"profit_matters":True,"liquidity_matters":False,"sector_multipliers":{"Semiconductor":2.5,"Hardware":1.8,"Software":1.3,"Cybersecurity":1.5,"Cloud Infra":2.0,"Streaming":0.8,"E-commerce":0.8,"Financials":0.3,"Banking":0.4,"Pharma":0.3,"Healthcare":0.3,"Insurance":0.3,"Oil Services":0.2,"Auto":0.3}},
    "Tariff War":      {"probability":0.12,"market_dd":-0.20,"valuation_matters":False,"mean_reversion":False,"profit_matters":False,"liquidity_matters":False,"sector_multipliers":{"Semiconductor":1.8,"Hardware":1.6,"Software":0.9,"Cybersecurity":0.8,"Cloud Infra":1.0,"Streaming":0.8,"E-commerce":1.2,"Financials":1.0,"Banking":1.0,"Pharma":0.6,"Healthcare":0.6,"Insurance":0.7,"Oil Services":1.3,"Auto":1.4}},
    "Recession":       {"probability":0.08,"market_dd":-0.30,"valuation_matters":True,"mean_reversion":True,"profit_matters":True,"liquidity_matters":True,"sector_multipliers":{"Semiconductor":1.3,"Hardware":1.3,"Software":1.1,"Cybersecurity":1.0,"Cloud Infra":1.2,"Streaming":0.9,"E-commerce":1.1,"Financials":1.4,"Banking":1.3,"Pharma":0.7,"Healthcare":0.7,"Insurance":0.9,"Oil Services":1.2,"Auto":1.3}},
    "Stagflation":     {"probability":0.03,"market_dd":-0.40,"valuation_matters":True,"mean_reversion":True,"profit_matters":True,"liquidity_matters":True,"sector_multipliers":{"Semiconductor":1.3,"Hardware":1.2,"Software":1.2,"Cybersecurity":1.2,"Cloud Infra":1.3,"Streaming":1.0,"E-commerce":1.1,"Financials":1.1,"Banking":1.1,"Pharma":0.8,"Healthcare":0.8,"Insurance":0.9,"Oil Services":0.7,"Auto":1.1}},
    "Mild Correction": {"probability":0.30,"market_dd":-0.10,"valuation_matters":False,"mean_reversion":True,"profit_matters":False,"liquidity_matters":False,"sector_multipliers":{"Semiconductor":1.1,"Hardware":1.1,"Software":1.0,"Cybersecurity":1.1,"Cloud Infra":1.1,"Streaming":1.0,"E-commerce":1.0,"Financials":1.0,"Banking":1.0,"Pharma":0.9,"Healthcare":0.9,"Insurance":0.9,"Oil Services":0.9,"Auto":1.0}},
    "Semi Downturn":   {"probability":0.15,"market_dd":-0.08,"valuation_matters":True,"mean_reversion":True,"profit_matters":False,"liquidity_matters":False,"sector_multipliers":{"Semiconductor":4.0,"Hardware":2.5,"Software":0.5,"Cybersecurity":0.5,"Cloud Infra":0.7,"Streaming":0.4,"E-commerce":0.4,"Financials":0.2,"Banking":0.3,"Pharma":0.2,"Healthcare":0.2,"Insurance":0.2,"Oil Services":0.3,"Auto":0.3}},
}

SECTOR_MAP = {"Semiconductors":"Semiconductor","Semiconductor Equipment & Materials":"Semiconductor","Software - Infrastructure":"Software","Software - Application":"Software","Information Technology Services":"Software","Internet Content & Information":"Software","Communication Equipment":"Hardware","Computer Hardware":"Hardware","Consumer Electronics":"Hardware","Electronic Components":"Hardware","Banks - Diversified":"Banking","Banks - Regional":"Banking","Capital Markets":"Financials","Financial Data & Stock Exchanges":"Financials","Credit Services":"Financials","Drug Manufacturers - General":"Pharma","Drug Manufacturers - Specialty & Generic":"Pharma","Biotechnology":"Pharma","Health Care Plans":"Healthcare","Health Care Providers":"Healthcare","Medical Instruments & Supplies":"Healthcare","Medical Devices":"Healthcare","Insurance - Diversified":"Insurance","Insurance - Specialty":"Insurance","Oil & Gas Equipment & Services":"Oil Services","Oil & Gas Drilling":"Oil Services","Auto Manufacturers":"Auto","Auto Parts":"Auto","Internet Retail":"E-commerce","Specialty Retail":"E-commerce","Entertainment":"Streaming","Leisure":"E-commerce"}
TICKER_SECTOR_OVERRIDE = {"RBRK":"Cybersecurity","CRWV":"Cloud Infra","GRAB":"E-commerce","SE":"E-commerce","MELI":"E-commerce","OSCR":"Insurance","SOFI":"Banking","CNC":"Healthcare","HROW":"Pharma","NFLX":"Streaming","SPOT":"Streaming","UBER":"Software","RDDT":"Software"}

# ======================================================================
#  SECTION 5: PRISM SCORING ENGINE
# ======================================================================

def pct_rank(series, ascending=True):
    return series.rank(pct=True, na_option="keep") * 100 if ascending else (1 - series.rank(pct=True, na_option="keep")) * 100 + (100 / len(series))

def score_growth(df, cfg):
    cap = cfg["growth_cap"]
    df["g26_capped"] = df["2026 Growth Rate"].clip(upper=cap)
    df["g27_capped"] = df["2027 Growth Rate"].clip(upper=cap)
    df["blended_growth"] = df["g26_capped"]*cfg["blend_2026"] + df["g27_capped"]*cfg["blend_2027"]
    df["growth_durability"] = np.where(df["g26_capped"]>0.05, (df["g27_capped"]/df["g26_capped"]).clip(0,3), 0)
    return pct_rank(df["g26_capped"])*0.40 + pct_rank(df["g27_capped"])*0.15 + pct_rank(df["blended_growth"])*0.30 + pct_rank(df["growth_durability"])*0.15

def score_value(df):
    df["is_profitable"] = (df["P/E.1"]>0) & (df["P/E.1"].notna())
    df["peg_proxy"] = np.where((df["P/E.1"]>0)&(df["blended_growth"]>0.01), df["P/E.1"]/(df["blended_growth"]*100), np.nan)
    peg_s = pct_rank(df["peg_proxy"], ascending=False)
    pe_s = pct_rank(df["P/E.1"].where(df["is_profitable"]), ascending=False)
    return np.where(df["is_profitable"], peg_s.fillna(30)*0.65+pe_s.fillna(30)*0.35, np.where(df["blended_growth"]>0.5,55,np.where(df["blended_growth"]>0.2,35,15)))

def score_momentum(df):
    df["52w_position"] = (df["Price"]-df["52 Week Low"])/(df["52 Week High"]-df["52 Week Low"])
    df["52w_range_width"] = (df["52 Week High"]-df["52 Week Low"])/df["52 Week Low"]
    s = pd.Series(100.0, index=df.index)
    s -= np.where(df["52w_position"]>0.90,20,np.where(df["52w_position"]>0.80,10,np.where(df["52w_position"]>0.70,5,0)))
    pf = df["peg_proxy"].fillna(1.0); s -= np.where(pf>3.0,15,np.where(pf>2.0,10,np.where(pf>1.5,5,0)))
    bf = df["Beta"].fillna(1.5); s -= np.where(bf>3.0,10,np.where(bf>2.5,7,np.where(bf>2.0,4,0)))
    s -= np.where(df["52w_range_width"]>5.0,10,np.where(df["52w_range_width"]>2.0,5,np.where(df["52w_range_width"]>1.0,2,0)))
    s += np.where(df["52w_position"]<0.30,10,np.where(df["52w_position"]<0.50,5,0))
    df["stretch_score"] = s.clip(0,100)
    return pct_rank(df["Average Outcome"])*0.45 + df["stretch_score"]*0.35 + pct_rank(df["52w_position"])*0.20

def score_resilience(df):
    df["growth_per_beta"] = df["blended_growth"]/df["Beta"].fillna(1.5).clip(lower=0.3)
    np2 = np.where(df["2026 Growth Rate"]<0,0,np.where(df["2026 Growth Rate"]<0.05,40,100))
    return pct_rank(df["growth_per_beta"])*0.45 + pct_rank(np.log10(df["Market Cap"].clip(lower=1e8)))*0.20 + np2*0.35

def run_prism(df, cfg):
    df["dim_growth"] = score_growth(df,cfg)
    df["dim_value"] = score_value(df)
    df["dim_momentum"] = score_momentum(df)
    df["dim_resilience"] = score_resilience(df)
    df["prism_score"] = (df["dim_growth"]*cfg["weight_growth"]+df["dim_value"]*cfg["weight_value"]+df["dim_momentum"]*cfg["weight_momentum"]+df["dim_resilience"]*cfg["weight_resilience"]).fillna(0)
    df["prism_rank"] = df["prism_score"].rank(ascending=False, method="min").astype(int)
    fp = pd.Series(0.0, index=df.index)
    fp += np.where(df["Beta"].fillna(0)>2.5,2.0,np.where(df["Beta"].fillna(0)>2.0,1.5,np.where(df["Beta"].fillna(0)>1.8,1.0,0)))
    fp += np.where(df["52w_position"]>0.90,2.0,np.where(df["52w_position"]>0.80,1.5,np.where(df["52w_position"]>0.70,1.0,0)))
    pp = np.where(df["peg_proxy"].fillna(np.nan)>3.0,1.5,np.where(df["peg_proxy"].fillna(np.nan)>1.5,1.0,np.where(df["peg_proxy"].fillna(0)>1.0,0.5,0)))
    prp = np.where(~df["is_profitable"],1.0,0)
    fp += np.maximum(pp,prp)
    rw = (df["52 Week High"]-df["52 Week Low"])/df["52 Week Low"]
    fp += np.where(rw>3.0,1.0,np.where(rw>1.5,0.5,0))
    df["fragile_points"] = fp
    df["fragile_flag"] = fp >= cfg.get("fragile_threshold",3.0)
    df["shock_sector"] = df["Industry"].map(SECTOR_MAP).fillna("default")
    for t,sc in TICKER_SECTOR_OVERRIDE.items():
        df.loc[df["Stock"]==t,"shock_sector"] = sc
    return df.sort_values("prism_rank")

# ======================================================================
#  SECTION 6: PORTFOLIO CONSTRUCTION
# ======================================================================

def build_portfolios(df, cfg):
    portfolios = {}
    n = len(df)
    nf = max(int(n*0.50),min(8,n))
    ds = df.sort_values("prism_score",ascending=False)
    pf = ds.head(nf).copy()
    portfolios["PRISM Full"] = pf["Stock"].tolist()
    frag = pf[pf["fragile_flag"]]["Stock"].tolist()
    base = pf[~pf["fragile_flag"]]["Stock"].tolist()
    nr = len(frag)
    cands = ds[(~ds["Stock"].isin(portfolios["PRISM Full"]))&(~ds["fragile_flag"])].head(max(nr+3,5)).copy()
    if len(cands)>0:
        cands["rv"] = cands["prism_score"]*0.5+cands["stretch_score"]*0.3+(100-cands["Beta"].fillna(1.5)*40)*0.2
        reps = cands.nlargest(nr,"rv")["Stock"].tolist()
    else:
        reps = []
    portfolios["PRISM Select"] = base + reps
    es = min(cfg["edge_size"],len(portfolios["PRISM Select"]))
    sdf = df[df["Stock"].isin(portfolios["PRISM Select"])].copy()
    ms = sdf["prism_score"].median()*0.85
    ee = sdf[(sdf["prism_score"]>=ms)&(sdf["Average Outcome"]>=0)].copy()
    if len(ee)<es:
        ee = sdf[sdf["Average Outcome"]>=0].copy()
    if len(ee)>0:
        def norm(s):
            r = s.max()-s.min()
            return (s-s.min())/r if r>0 else 0.5
        eq = ee["Average Outcome"]*(ee["stretch_score"]/100)
        ee["edge_score"] = norm(ee["prism_score"])*0.40+norm(eq)*0.25+norm(ee["blended_growth"])*0.25+norm(1-ee["52w_position"])*0.10
        portfolios["PRISM Edge"] = ee.nlargest(es,"edge_score")["Stock"].tolist()
    else:
        portfolios["PRISM Edge"] = sdf.nlargest(es,"prism_score")["Stock"].tolist()
    custom = [s for s in portfolios["PRISM Edge"] if s not in cfg["force_exclude"]]
    for s in cfg["force_include"]:
        if s not in custom and s in df["Stock"].values: custom.append(s)
    portfolios["Custom"] = custom
    return portfolios, frag, reps

# ======================================================================
#  SECTION 7: SHOCK STRESS TEST
# ======================================================================

def safe_get(row, key, default):
    val = row.get(key, default)
    return default if pd.isna(val) else val

def shock_stock(row, scenario):
    bd = scenario["market_dd"]
    bm = 0.5+0.5*safe_get(row,"Beta",1.5)
    sm = scenario["sector_multipliers"].get(safe_get(row,"shock_sector","default"),1.0)
    if scenario.get("valuation_matters"):
        pe = safe_get(row,"P/E.1",80); pe = pe if pe>0 else 80
        vm = 1.3 if pe>80 else (1.15 if pe>50 else (1.0 if pe>30 else 0.85))
    else: vm = 1.0
    if scenario.get("liquidity_matters"):
        mc = safe_get(row,"Market Cap",1e10)/1e9
        szm = 1.3 if mc<10 else (1.1 if mc<50 else 1.0)
    else: szm = 1.0
    pm = (0.8+0.4*safe_get(row,"52w_position",0.5)) if scenario.get("mean_reversion") else 1.0
    prm = 1.3 if (scenario.get("profit_matters") and not safe_get(row,"is_profitable",True)) else 1.0
    return max(bd*bm*sm*vm*szm*pm*prm, -0.95)

def run_shock(df, portfolios):
    results = {}
    for pn, sl in portfolios.items():
        pdf = df[df["Stock"].isin(sl)]
        n = len(pdf)
        if n==0: continue
        r = {"N":n,"Growth":pdf["blended_growth"].mean(),"Beta":pdf["Beta"].fillna(1.5).mean(),
             "Stretch":pdf["stretch_score"].mean(),"52w":pdf["52w_position"].mean(),"R/R":pdf["Average Outcome"].mean()}
        pp = pdf[pdf["peg_proxy"].notna()&(pdf["peg_proxy"]>0)]
        r["PEG"] = pp["peg_proxy"].mean() if len(pp)>0 else np.nan
        r["Semi%"] = pdf["shock_sector"].eq("Semiconductor").mean()*100
        r["PreProfit%"] = (~pdf["is_profitable"]).mean()*100
        r["G/B"] = r["Growth"]/r["Beta"] if r["Beta"]>0 else 0
        for sn,sp in SHOCK_SCENARIOS.items():
            r[sn] = np.nanmean([shock_stock(row,sp) for _,row in pdf.iterrows()])*100
        r["SHOCK Score"] = sum(SHOCK_SCENARIOS[s]["probability"]*r[s] for s in SHOCK_SCENARIOS)
        gp = r["Growth"]*100
        r["Edge Ratio"] = gp/abs(r["SHOCK Score"]) if r["SHOCK Score"]!=0 else 0
        dd = r["SHOCK Score"]/100
        rn = -dd/(1+dd)*100
        r["Recovery Needed"] = rn
        r["1Y Recovery"] = "YES" if rn<gp else ("MAYBE" if rn<gp*2 else "NO")
        results[pn] = r
    return results

# ======================================================================
#  SECTION 8: REPORT GENERATION
# ======================================================================

def print_header(text, width=120, char="="):
    print(f"\n{char*width}")
    print(f"  {text}")
    print(f"{char*width}")

def generate_report(df, portfolios, fragile_stocks, replacements, shock_results, cfg):
    print_header("PRISM v2.3 — Fixed Multi-Source")
    print(f"  Date: {datetime.now().strftime('%Y-%m-%d %H:%M')} | Stocks: {len(df)}")
    print(f"  Weights: G:{cfg['weight_growth']:.0%} V:{cfg['weight_value']:.0%} M:{cfg['weight_momentum']:.0%} R:{cfg['weight_resilience']:.0%}")
    print_header("PRISM RANKINGS - Top 30")
    top30 = df.head(30)
    print(f"\n {'Rk':>3} {'Stock':>6} {'Score':>6} | {'Grw':>4} {'Val':>4} {'Mom':>4} {'Res':>4} | {'26E':>5} {'27E':>5} {'Blnd':>5} | {'P/E':>6} {'PEG':>6} {'Beta':>5} | {'52w%':>5} {'Stch':>4} {'R/R':>6} | {'Frag':>9}")
    print(f" {'-'*112}")
    for _, r in top30.iterrows():
        pe_s = f"{r['P/E.1']:6.1f}" if pd.notna(r['P/E.1']) and r['P/E.1']>0 else "   N/A"
        pg_s = f"{r['peg_proxy']:6.2f}" if pd.notna(r['peg_proxy']) else "   N/A"
        b_s = f"{r['Beta']:5.2f}" if pd.notna(r['Beta']) else "  N/A"
        fl = f"  F({r['fragile_points']:.1f})" if r['fragile_flag'] else f"  ({r['fragile_points']:.1f})"
        print(f" {int(r['prism_rank']):3d} {r['Stock']:>6} {r['prism_score']:6.1f} | {r['dim_growth']:4.0f} {r['dim_value']:4.0f} {r['dim_momentum']:4.0f} {r['dim_resilience']:4.0f} | {r['2026 Growth Rate']:4.0%} {r['2027 Growth Rate']:4.0%} {r['blended_growth']:4.0%} | {pe_s} {pg_s} {b_s} | {r['52w_position']:4.0%} {r['stretch_score']:4.0f} {r['Average Outcome']:6.3f} | {fl}")
    if fragile_stocks:
        print_header("FRAGILE FLAGS", char="-")
        print(f"  Flagged ({len(fragile_stocks)}): {fragile_stocks}")
        if replacements: print(f"  Replacements ({len(replacements)}): {replacements}")
    for pname, sl in portfolios.items():
        nn = len(sl); wt = 100/nn if nn>0 else 0
        print_header(f"{pname} - {nn} stocks x {wt:.1f}%", char="-")
        pdf = df[df["Stock"].isin(sl)].sort_values("prism_score",ascending=False)
        for _, r in pdf.iterrows():
            pe_s = f"{r['P/E.1']:.0f}x" if pd.notna(r['P/E.1']) and r['P/E.1']>0 else " N/A"
            b_s = f"{r['Beta']:.2f}" if pd.notna(r['Beta']) else " N/A"
            print(f"    {r['Stock']:>6} | Score {r['prism_score']:5.1f} | G:{r['blended_growth']:4.0%} | PE:{pe_s:>5} | B:{b_s:>5} | 52w:{r['52w_position']:3.0%} | Str:{r['stretch_score']:3.0f} | R/R:{r['Average Outcome']:.3f} | {r['shock_sector']}")
    print_header("SHOCK STRESS TEST & EDGE RATIO")
    pnames = list(shock_results.keys())
    print(f"\n {'Metric':20s}", end="")
    for pn in pnames: print(f" | {pn:>14}", end="")
    print(f" | {'SPY':>8}")
    print(f" {'-'*(25+17*len(pnames))}")
    for label, key, fmt in [("Positions","N","{:.0f}"),("Blended Growth","Growth","{:.0%}"),("PEG","PEG","{:.2f}x"),("Beta","Beta","{:.2f}"),("Stretch","Stretch","{:.0f}"),("52w Position","52w","{:.0%}"),("R/R Ratio","R/R","{:.3f}"),("Growth/Beta","G/B","{:.3f}"),("Semi%","Semi%","{:.0f}%"),("Pre-Profit%","PreProfit%","{:.0f}%")]:
        print(f" {label:20s}", end="")
        for pn in pnames:
            v = shock_results[pn].get(key,np.nan)
            s = fmt.format(v) if pd.notna(v) else "N/A"
            print(f" | {s:>14}", end="")
        print(f" | {'':>8}")
    print()
    for sn in SHOCK_SCENARIOS:
        print(f" {sn:20s}", end="")
        vals = [shock_results[pn].get(sn,0) for pn in pnames]; best = max(vals)
        for v in vals:
            m = " <<" if v==best and len(pnames)>1 else ""
            print(f" | {v:+10.0f}%{m:>3}", end="")
        print(f" | {SHOCK_SCENARIOS[sn]['market_dd']*100:+7.0f}%")
    print()
    spy_s = sum(s["probability"]*s["market_dd"]*100 for s in SHOCK_SCENARIOS.values())
    print(f" {'SHOCK Score':20s}", end="")
    for pn in pnames: print(f" | {shock_results[pn]['SHOCK Score']:+10.1f}%    ", end="")
    print(f" | {spy_s:+7.1f}%")
    print(f" {'* EDGE RATIO *':20s}", end="")
    evs = [shock_results[pn]["Edge Ratio"] for pn in pnames]; be = max(evs)
    for i,pn in enumerate(pnames):
        m = " <<" if evs[i]==be and len(pnames)>1 else ""
        print(f" | {evs[i]:10.2f}x{m:>3}", end="")
    print(f" | {10/abs(spy_s):7.2f}x" if spy_s!=0 else " |    N/A")
    print(f"\n {'Recovery (1Y)':20s}", end="")
    for pn in pnames: print(f" | {shock_results[pn]['1Y Recovery']:>14}", end="")
    print()
    bp = max(shock_results, key=lambda x: shock_results[x]["Edge Ratio"]); b = shock_results[bp]
    print_header(f"RECOMMENDATION: {bp}")
    print(f"  Edge Ratio: {b['Edge Ratio']:.2f}x | Growth: {b['Growth']:.0%} | SHOCK: {b['SHOCK Score']:+.1f}% | Stocks: {b['N']}")
    print(f"  Holdings: {portfolios[bp]}")

# ======================================================================
#  SECTION 9: EXPORT
# ======================================================================

def export_csvs(df, portfolios, shock_results, cfg):
    import os
    out = cfg["output_dir"]
    cols = ["Stock","prism_rank","prism_score","dim_growth","dim_value","dim_momentum","dim_resilience","2026 Growth Rate","2027 Growth Rate","blended_growth","P/E.1","peg_proxy","Beta","52w_position","stretch_score","Average Outcome","Market Cap","fragile_flag","fragile_points","shock_sector","Industry","Price","Target Mean"]
    vc = [c for c in cols if c in df.columns]
    df[vc].sort_values("prism_rank").to_csv(os.path.join(out,"prism_scored_watchlist.csv"),index=False)
    rows = []
    for pn,sl in portfolios.items():
        for s in sl: rows.append({"Portfolio":pn,"Stock":s,"Weight":100/len(sl)})
    pd.DataFrame(rows).to_csv(os.path.join(out,"prism_portfolios.csv"),index=False)
    rr = [{"Portfolio":pn,**m} for pn,m in shock_results.items()]
    pd.DataFrame(rr).to_csv(os.path.join(out,"prism_report.csv"),index=False)
    print(f"\n  Exported: prism_scored_watchlist.csv, prism_portfolios.csv, prism_report.csv")

# ======================================================================
#  SECTION 10: RUN
# ======================================================================

df, quality_df = fetch_all_data(TICKERS, CONFIG, FMP_API_KEY, FINNHUB_API_KEY)
print("\n  Running PRISM scoring engine...")
df = run_prism(df, CONFIG)
print("  Building portfolios...")
portfolios, fragile, reps = build_portfolios(df, CONFIG)
print("  Running SHOCK stress tests...")
shock = run_shock(df, portfolios)
generate_report(df, portfolios, fragile, reps, shock, CONFIG)
export_csvs(df, portfolios, shock, CONFIG)
quality_df.to_csv("prism_data_quality.csv", index=False)
print("  Exported: prism_data_quality.csv")
print("\n  PRISM v2.3 complete!")


  PRISM v2.3 — Multi-Source Data Fetch
  Tickers: 30 | Sources: FMP (stable) + Finnhub + yfinance

  ⚠️  FMP returned nothing for AVGO (fail 1/3)
  ⚠️  Finnhub returned nothing for AVGO (fail 1/3)
  [ 1/30]   AVGO | G26:  +66% [ 📊Y] | G27:  +57% [ 📊Y] | PE:  27.7 | B: 1.26
  ⚠️  FMP returned nothing for MU (fail 2/3)
  ⚠️  Finnhub returned nothing for MU (fail 2/3)
  [ 2/30]     MU | G26:   +6% [ 📊Y] | G27:  +71% [ 📊Y] | PE:   6.3 | B: 1.61
  ⚠️  Finnhub returned nothing for TSM (fail 3/3)
  ❌ Finnhub failed on first 3 tickers — disabling (likely premium-only)
  [ 3/30]    TSM | G26:  +36% [ 📊Y] | G27:  +24% [ 📊Y] | PE:  23.4 | B: 1.28
  ❌ FMP failed on first 3 tickers — disabling FMP
  [ 4/30]   ASML | G26:  +20% [ 📊Y] | G27:  +27% [ 📊Y] | PE:  44.3 | B: 1.43
  [ 5/30] 000660.KS | G26: +229% [ 📊Y] | G27:  +16% [ 📊Y] | PE:   4.4 | B: 1.72
  [ 6/30]    APP | G26:  +44% [ 📊Y] | G27:  +33% [ 📊Y] | PE:  24.6 | B: 2.50
  [ 7/30]   RBRK | G26:  +18% [ 📊Y] | G27: +237% [ 📊Y] | PE: 295.2 | B:

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"No fundamentals data found for symbol: BRK.B"}}}


  [24/30]  BRK.B | G26:  +17% [ 📊Y] | G27:  +17% [ 📊Y] | PE:   N/A | B:  N/A
  [25/30]   FICO | G26:  +40% [ 📊Y] | G27:  +27% [ 📊Y] | PE:  26.0 | B: 1.28
  [26/30]      V | G26:  +12% [ 📊Y] | G27:  +13% [ 📊Y] | PE:  23.4 | B: 0.79
  [27/30]     MA | G26:  +15% [ 📊Y] | G27:  +16% [ 📊Y] | PE:  25.2 | B: 0.84
  [28/30]     NU | G26:  +40% [ 📊Y] | G27:  +30% [ 📊Y] | PE:  16.8 | B: 1.11
  [29/30]   CVNA | G26:  -12% [ 📊Y] | G27:  +44% [ 📊Y] | PE:  45.5 | B: 3.67
  [30/30]   NBIS | G26:   +4% [ ❓D] | G27:  +71% [ 📊Y] | PE:-158.0 | B: 1.16

  DATA QUALITY SCORECARD
  Sources active: FMP=NO (disabled) | Finnhub=NO (disabled) | yfinance=YES
  2026 Growth: ✅HIGH:0 ⚠MED:0 🚩LOW:0 📊YF:29 ❓DEFAULT:1 🔧OVR:0 | 0% reliable
  2027 Growth: ✅HIGH:0 ⚠MED:0 🚩LOW:0 📊YF:30 ❓DEFAULT:0 🔧OVR:0 | 0% reliable

  ⚠️  Stocks needing overrides: ['NBIS']

  Running PRISM scoring engine...
  Building portfolios...
  Running SHOCK stress tests...

  PRISM v2.3 — Fixed Multi-Source
  Date: 2026-04-03 07:48 | Stocks: 30
 

In [ ]:
from google.colab import files
files.download("prism_scored_watchlist.csv")
files.download("prism_portfolios.csv")
files.download("prism_report.csv")
files.download("prism_data_quality.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>